In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments, DistilBertForQuestionAnswering, DistilBertTokenizerFast
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import gc
from transformers import EarlyStoppingCallback

In [2]:
dataset_path= 'Model_dataset/synthetic_question_ans_data.csv'

# q type classifer,
q_type_model_name= 'distilbert-base-uncased'
q_type_model_result= '.temp/model_results/q_types_model_lite_results'
q_type_model= 'temp/model/fine_tuned_question_classifier_model_lite'

# qa model,
qa_type_model_name= 'distilbert-base-uncased'
qa_type_model_result= '.temp/model_results/qa_types_model_lite_results'
qa_type_model= 'temp/model/fine_tuned_question_answer_model_lite'


# Question type classsifier

### Preprocessing

In [3]:
df= pd.read_csv(dataset_path)
df= df[["question", "question_type"]]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       499 non-null    object
 1   question_type  499 non-null    object
dtypes: object(2)
memory usage: 7.9+ KB


In [4]:
df["question_type"].unique()

array(['general question', 'tech specific experience', 'notice period',
       'expected ctc', 'contact information', 'current ctc', 'name'],
      dtype=object)

In [5]:
df.drop_duplicates(inplace= True)

In [6]:
# adding labels
label_mapping = {'contact information': 0,
    'tech specific experience': 1,
    'general question': 2,
    'notice period': 3,
    'current ctc': 4,
    'name': 5,
    'expected ctc': 6
    }

df['label'] = df['question_type'].map(label_mapping)

# droping unused column
df.drop('question_type', axis=1,  inplace= True)
df.head()

,question,label
0,Have you worked with RESTful APIs?,2
1,How do you handle data deduplication in large ...,1
2,Do you have experience in building microservices?,2
3,What are your strategies for optimizing API pe...,1
4,What database optimization techniques have you...,2


In [7]:
df= df.sample(frac=1).reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 443 entries, 0 to 442
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  443 non-null    object
 1   label     443 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 7.1+ KB


### Retraing Preparations:

In [8]:
#convert to hugging face dataset
dataset= Dataset.from_pandas(df)

#Split the data into train and test sets (80-20 split)
dataset_split = dataset.train_test_split(test_size=0.2)

# Access train and test splits
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

In [9]:
tokenizer = DistilBertTokenizer.from_pretrained(q_type_model_name)

In [10]:
def tokenize_function(examples):
    return tokenizer(examples['question'], padding= "max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set the format to PyTorch tensors
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/354 [00:00<?, ? examples/s]

Map:   0%|          | 0/89 [00:00<?, ? examples/s]

In [11]:
# Mapping lebel and id
id2label = {v: k for k, v in label_mapping.items()}  # Map IDs to label names
label2id = {k: v for k, v in label_mapping.items()}  # Map label names to IDs

### Retraning

In [12]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_name,
        num_labels=7,
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


7


In [13]:
training_args = TrainingArguments(
    output_dir= q_type_model_result,           # Output directory
    eval_strategy="epoch",     # Evaluate after a specific number of steps
    save_strategy="epoch",           # Save the model after a specific number of steps
    learning_rate= 3e-5,
    num_train_epochs= 10,             # Number of training epochs
    per_device_train_batch_size= 16,   # Batch size per device during training
    per_device_eval_batch_size= 16,    # Batch size per device during evaluation
    gradient_accumulation_steps=2,
    logging_dir='./logs',            # Directory for storing logs
    logging_steps=10,                # Log every 10 steps
    load_best_model_at_end=True,     # Required for EarlyStoppingCallback
    # use_cpu=True                     # Force CPU usage (but TrainingArguments doesn’t support this; see notes below)
)

trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.783400,1.672003,0.359551
2,1.493000,1.382755,0.370787
3,1.218800,1.137673,0.764045
4,1.051300,0.946751,0.887640
5,0.781100,0.836270,0.865169
6,0.740600,0.716387,0.887640
7,0.608600,0.679778,0.876404
8,0.558400,0.646454,0.887640
9,0.488100,0.629335,0.898876


TrainOutput(global_step=110, training_loss=0.9275738586078991, metrics={'train_runtime': 1891.096, 'train_samples_per_second': 1.872, 'train_steps_per_second': 0.058, 'total_flos': 430557434112000.0, 'train_loss': 0.9275738586078991, 'epoch': 9.173913043478262})

### Model evaluation and Saving

In [14]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.6293349266052246,
 'eval_accuracy': 0.898876404494382,
 'eval_runtime': 32.2675,
 'eval_samples_per_second': 2.758,
 'eval_steps_per_second': 0.186,
 'epoch': 9.173913043478262}

In [15]:
trainer.save_model(q_type_model)
tokenizer.save_pretrained(q_type_model)

('model/fine_tuned_question_classifier_model_lite/tokenizer_config.json',
 'model/fine_tuned_question_classifier_model_lite/special_tokens_map.json',
 'model/fine_tuned_question_classifier_model_lite/vocab.txt',
 'model/fine_tuned_question_classifier_model_lite/added_tokens.json')

# Question- Answer model

### Preprocessing

In [16]:
df= pd.read_csv(dataset_path)
df = df[~df["question_type"].isin(['expected ctc', 'current ctc', 'notice period'])]

In [17]:
df.drop_duplicates(inplace= True)
df= df.sample(frac=1).reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368 entries, 0 to 367
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       368 non-null    object
 1   question_type  368 non-null    object
 2   answer         330 non-null    object
dtypes: object(3)
memory usage: 8.8+ KB


In [18]:
df["question_type"].unique()

array(['name', 'tech specific experience', 'general question',
       'contact information'], dtype=object)

In [19]:
# loading CV for context
file= open("Model_dataset/cv_personal_info.txt","r")
cv_personal_info_text= file.read()
file.close()

# loading CV for context
file= open("Model_dataset/cv_workand_skill_info.txt","r")
cv_workand_skill_info_text= file.read()
file.close()

In [20]:
df["context"]= cv_personal_info_text
df.loc[df['question_type'] == 'tech specific experience', 'context'] = cv_workand_skill_info_text

In [21]:
df.head()

,question,question_type,answer,context
0,Do you have any professional alias or pen name?,name,Manab Boro,"Personal Information\nManab Boro\nGuwahati,Ass..."
1,What are your preferred CI/CD tools?,tech specific experience,Used GitHub Actions and Jenkins.,WORK EXPERINCE\nTriedatum Inc\n\tSoftware Engi...
2,How do you ensure idempotency in data processi...,tech specific experience,Used unique keys and transactional operations.,WORK EXPERINCE\nTriedatum Inc\n\tSoftware Engi...
3,What is your experience level with Apache Spark?,tech specific experience,Intermediate,WORK EXPERINCE\nTriedatum Inc\n\tSoftware Engi...
4,Have you used Apache Airflow in your projects?,tech specific experience,"Yes, used it for business report automation",WORK EXPERINCE\nTriedatum Inc\n\tSoftware Engi...


In [22]:
df.drop('question_type', axis=1, inplace= True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368 entries, 0 to 367
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  368 non-null    object
 1   answer    330 non-null    object
 2   context   368 non-null    object
dtypes: object(3)
memory usage: 8.8+ KB


In [23]:
df.head()

,question,answer,context
0,Do you have any professional alias or pen name?,Manab Boro,"Personal Information\nManab Boro\nGuwahati,Ass..."
1,What are your preferred CI/CD tools?,Used GitHub Actions and Jenkins.,WORK EXPERINCE\nTriedatum Inc\n\tSoftware Engi...
2,How do you ensure idempotency in data processi...,Used unique keys and transactional operations.,WORK EXPERINCE\nTriedatum Inc\n\tSoftware Engi...
3,What is your experience level with Apache Spark?,Intermediate,WORK EXPERINCE\nTriedatum Inc\n\tSoftware Engi...
4,Have you used Apache Airflow in your projects?,"Yes, used it for business report automation",WORK EXPERINCE\nTriedatum Inc\n\tSoftware Engi...


### Retraing Preparations:

In [24]:
model = DistilBertForQuestionAnswering.from_pretrained(qa_type_model_name)
tokenizer = DistilBertTokenizerFast.from_pretrained(qa_type_model_name)

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
dataset= Dataset.from_pandas(df)


# Split the data into train and test sets (80-20 split)
dataset_split = dataset.train_test_split(test_size=0.2)

# Access train and test splits
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

In [26]:
# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["question"], examples["context"], truncation=True, padding=True, return_tensors="pt")


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/294 [00:00<?, ? examples/s]

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

In [27]:
def find_answer_positions(examples):
    start_positions = []
    end_positions = []
    
    for i, (context, answer) in enumerate(zip(examples['context'], examples['answer'])):
        if answer:  # Check if answer is not None or empty
            start_idx = context.find(answer)
            if start_idx != -1:  # If the answer is found in the context
                end_idx = start_idx + len(answer)
                start_positions.append(start_idx)
                end_positions.append(end_idx)
            else:
                start_positions.append(-1)  # If the answer is not found, set start and end as -1
                end_positions.append(-1)
        else:
            start_positions.append(-1)  # If no answer, set start and end as -1
            end_positions.append(-1)
    
    examples['start_positions'] = start_positions
    examples['end_positions'] = end_positions
    return examples


train_dataset = train_dataset.map(find_answer_positions, batched=True)
test_dataset = test_dataset.map(find_answer_positions, batched=True)

Map:   0%|          | 0/294 [00:00<?, ? examples/s]

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

### Train the model

In [28]:
# Set up the training arguments
training_args = TrainingArguments(
    output_dir= qa_type_model_result,           # Output directory
    eval_strategy="epoch",     # Evaluate after a specific number of steps
    save_strategy="epoch",           # Save the model after a specific number of steps
    learning_rate= 3e-5,
    num_train_epochs= 10,             # Number of training epochs
    per_device_train_batch_size= 16,   # Batch size per device during training
    per_device_eval_batch_size= 16,    # Batch size per device during evaluation
    gradient_accumulation_steps=2,
    logging_dir='./logs',            # Directory for storing logs
    logging_steps=10,                # Log every 10 steps
    load_best_model_at_end=True,     # Required for EarlyStoppingCallback
    # use_cpu=True    
    weight_decay=0.01,
)

# Set up the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,  # Use a separate validation dataset if available
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Fine-tune the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.343200,2.113347
2,1.352600,0.861300
3,0.790100,0.718113
4,0.685500,0.643022
5,0.640300,0.620328
6,0.592800,0.596872
7,0.598400,0.571933
8,0.541200,0.544709


RuntimeError: [enforce fail at inline_container.cc:603] . unexpected pos 169519872 vs 169519760

### Model evaluation and Saving

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model)
tokenizer.save_pretrained(q_type_model)